In [0]:
SELECT * FROM com_edp_prd.com_raw.kom_providers LIMIT 1001;

In [0]:
SELECT * FROM
(select DISTINCT PRIMARY_SPECIALTY AS SPECIALTY from com_raw.kom_providers
WHERE PROVIDER_TYPE = 'INDIVIDUAL'
UNION
select DISTINCT SECONDARY_SPECIALTY AS SPECIALTY from com_raw.kom_providers
WHERE PROVIDER_TYPE = 'INDIVIDUAL')
WHERE specialty ilike '%pharma%'

In [0]:
select * from com_edp_prd.com_raw.vod_references;

In [0]:
select * from com_edp_prd.com_raw.vod_hcp;
-- where hcp_type_cda__v ilike '%phar%';

In [0]:
select distinct primary_specialty_group__v, specialty_1__v, spec_1_cda__v from com_edp_prd.com_raw.vod_hcp
where specialty_1__v = 'PA';

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE AS
SELECT *
FROM (
    -- Medical (Elaprase NDC)
    SELECT DISTINCT
        PATIENT_ID                                  AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI)      AS HCP_NPI,
        BILLING_NPI                                AS HCO_NPI,
        NDC11                                       AS CODE,
        MEDICAL_EVENT_ID                            AS EVENT_ID,
        SERVICE_DATE                                AS FILL_DATE,
        PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
        KH_PLAN_ID                                  AS KH_PLAN,
        null                                        AS PHARMACY_CHANNEL,
        'MEDICAL_EVENTS'                            AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    -- Pharmacy (Elaprase NDC | Paid only)
    SELECT DISTINCT
        PATIENT_ID                                  AS PATIENT_ID,
        PRESCRIBER_NPI                              AS HCP_NPI,
        PHARMACY_NPI                                AS HCO_NPI,
        NDC11                                       AS CODE,
        PHARMACY_EVENT_ID                           AS EVENT_ID,
        FILL_DATE                                   AS FILL_DATE,
        NULL                                        AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        PHARMACY_CHANNEL                            AS PHARMACY_CHANNEL,
        'PHARMACY_EVENTS'                           AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    -- Medical (Elaprase Procedure codes)
    SELECT DISTINCT
        PATIENT_ID                                  AS PATIENT_ID,
        RENDERING_NPI                               AS HCP_NPI,
        BILLING_NPI                                AS HCO_NPI,
        PROCEDURE_CODE                              AS CODE,
        MEDICAL_EVENT_ID                            AS EVENT_ID,
        SERVICE_DATE                                AS FILL_DATE,
        PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
        KH_PLAN_ID                                  AS KH_PLAN,
        null                                        AS PHARMACY_CHANNEL,
        'MEDICAL_EVENTS'                            AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
    )
) t
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';

In [0]:
CREATE OR REPLACE TEMP VIEW MPSII_HCP_Territory AS
SELECT
  a.*,
  c.territory_name,
  c.territory_id,
  c.region_name,
  c.region_id
FROM mpsii_treatment_table a
LEFT JOIN com_edp_prd.com_raw.kom_providers b
  ON a.hcp_npi = b.npi
 AND b.provider_type = 'INDIVIDUAL'
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping c
  ON TRY_CAST(
       SUBSTR(REGEXP_REPLACE(COALESCE(b.PROVIDER_ZIP, ''), '[^0-9]', ''), 1, 5)
       AS BIGINT
     ) = c.zipcode;
SELECT * FROM MPSII_HCP_Territory;


In [0]:
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE AS
WITH base AS (
  SELECT *
  FROM (
      -- Medical (Elaprase NDC)
      SELECT DISTINCT
          PATIENT_ID                                  AS PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI)      AS HCP_NPI,
          BILLING_NPI                                AS HCO_NPI,
          NDC11                                       AS CODE,
          MEDICAL_EVENT_ID                            AS EVENT_ID,
          SERVICE_DATE                                AS FILL_DATE,
          PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
          KH_PLAN_ID                                  AS KH_PLAN,
          NULL                                        AS PHARMACY_CHANNEL,
          'MEDICAL_EVENTS'                            AS TABLE_NAME
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')

      UNION

      -- Pharmacy (Elaprase NDC | Paid only)
      SELECT DISTINCT
          PATIENT_ID                                  AS PATIENT_ID,
          PRESCRIBER_NPI                              AS HCP_NPI,
          PHARMACY_NPI                                AS HCO_NPI,
          NDC11                                       AS CODE,
          PHARMACY_EVENT_ID                           AS EVENT_ID,
          FILL_DATE                                   AS FILL_DATE,
          NULL                                        AS PLACE_OF_SERVICE,
          COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
          PHARMACY_CHANNEL                            AS PHARMACY_CHANNEL,
          'PHARMACY_EVENTS'                           AS TABLE_NAME
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'

      UNION

      -- Medical (Elaprase Procedure codes)
      SELECT DISTINCT
          PATIENT_ID                                  AS PATIENT_ID,
          RENDERING_NPI                               AS HCP_NPI,
          BILLING_NPI                                AS HCO_NPI,
          PROCEDURE_CODE                              AS CODE,
          MEDICAL_EVENT_ID                            AS EVENT_ID,
          SERVICE_DATE                                AS FILL_DATE,
          PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
          KH_PLAN_ID                                  AS KH_PLAN,
          NULL                                        AS PHARMACY_CHANNEL,
          'MEDICAL_EVENTS'                            AS TABLE_NAME
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN (
          '99601','99602','96365','96366','J1743','S9357','S9379',
          '38206','38230','38232','38240','38241','38242','38243','38250'
      )
  ) t
  WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30'
),
prov AS (
  -- One row per NPI (helps avoid row multiplication if kom_providers has multiple rows per NPI)
  SELECT
    TRIM(CAST(NPI AS STRING)) AS npi_str,
    MAX(
      TRY_CAST(
        SUBSTR(REGEXP_REPLACE(COALESCE(PROVIDER_ZIP, ''), '[^0-9]', ''), 1, 5) AS BIGINT
      )
    ) AS zip5_int
  FROM com_edp_prd.com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
  GROUP BY TRIM(CAST(NPI AS STRING))
)
SELECT
  b.*,
  z.territory_name,
  z.territory_id,
  z.region_name,
  z.region_id
FROM base b
LEFT JOIN prov p
  ON TRIM(CAST(b.HCP_NPI AS STRING)) = p.npi_str
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
  ON p.zip5_int = z.zipcode;
select * from mpsii_treatment_table

In [0]:
%sql
-- ============================================================
-- 2. TOTAL LIVES BASE (ANY medical + PAID pharmacy) + TERRITORY
-- ============================================================
CREATE OR REPLACE TEMP VIEW total_lives AS
WITH base AS (
  SELECT DISTINCT
      PATIENT_ID                                  AS total_lives,
      COALESCE(RENDERING_NPI, REFERRING_NPI)      AS HCP_NPI,
      BILLING_NPI                                AS HCO_NPI,
      MEDICAL_EVENT_ID                            AS EVENT_ID,
      SERVICE_DATE                                AS FILL_DATE,
      PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
      KH_PLAN_ID                                  AS KH_PLAN,
      NULL                                        AS PHARMACY_CHANNEL,
      'MEDICAL_EVENTS'                            AS TABLE_NAME
  FROM com_edp_prd.com_raw.kom_medical_events
  WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '2025-11-30'

  UNION

  SELECT DISTINCT
      PATIENT_ID                                  AS total_lives,
      PRESCRIBER_NPI                              AS HCP_NPI,
      PHARMACY_NPI                                AS HCO_NPI,
      PHARMACY_EVENT_ID                           AS EVENT_ID,
      FILL_DATE                                   AS FILL_DATE,
      NULL                                        AS PLACE_OF_SERVICE,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      PHARMACY_CHANNEL                            AS PHARMACY_CHANNEL,
      'PHARMACY_EVENTS'                           AS TABLE_NAME
  FROM com_edp_prd.com_raw.kom_pharmacy_events
  WHERE TRANSACTION_RESULT = 'PAID'
    AND FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30'
),
prov AS (
  -- One row per NPI to avoid row explosion
  SELECT
    TRIM(CAST(NPI AS STRING)) AS npi_str,
    MAX(
      TRY_CAST(
        SUBSTR(REGEXP_REPLACE(COALESCE(PROVIDER_ZIP, ''), '[^0-9]', ''), 1, 5) AS BIGINT
      )
    ) AS zip5_int
  FROM com_edp_prd.com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
  GROUP BY TRIM(CAST(NPI AS STRING))
)
SELECT
  b.*,
  z.territory_name,
  z.territory_id,
  z.region_name,
  z.region_id
FROM base b
LEFT JOIN prov p
  ON TRIM(CAST(b.HCP_NPI AS STRING)) = p.npi_str
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
  ON p.zip5_int = z.zipcode;
select * from total_lives